# Configuração do Projeto

Notebook de configuração reutilizável para Bronze, Silver e Gold.

In [0]:
%pip install -q python-dotenv azure-storage-file-datalake azure-identity deltalake pyarrow

In [0]:
# %run ../utils/utils

## Imports e leitura do `.env`

In [0]:
from dotenv import load_dotenv
from pathlib import Path
from io import BytesIO
import os
import uuid

# Removidas as bibliotecas pyarrow e deltalake, pois agora usamos Spark nativo!
from azure.identity import ClientSecretCredential
from azure.storage.filedatalake import DataLakeServiceClient

possiveis_envs = [
    Path('../.env'),
    Path('../../.env'),
    Path('/Workspace/.env'),
    Path.cwd() / '.env',
]

env_carregado = False
for env_path in possiveis_envs:
    if env_path.exists():
        load_dotenv(env_path)
        env_carregado = True
        print(f'.env carregado de: {env_path}')
        break

if not env_carregado:
    load_dotenv()
    print('Tentando carregar .env pelo caminho padrão.')

# Coleta as variáveis
CLIENT_ID = os.getenv('client_id') or os.getenv('CLIENT_ID')
TENANT_ID = os.getenv('tenant_id') or os.getenv('TENANT_ID')
CLIENT_SECRET = os.getenv('client_secret') or os.getenv('CLIENT_SECRET')
STORAGE_ACCOUNT_NAME = os.getenv('storage_account_name') or os.getenv('STORAGE_ACCOUNT_NAME') or "internshipdatalake"

CONTAINER_RAW = os.getenv('container_name') or os.getenv('CONTAINER_NAME') or 'raw'
CONTAINER_SQUAD1 = os.getenv('container_squad1') or os.getenv('CONTAINER_SQUAD1') or 'squad1'
RAW_FOLDER = os.getenv('raw_folder') or os.getenv('RAW_FOLDER') or 'real-time-data'


JDBC_HOSTNAME = os.getenv('jdbc_hostname') or os.getenv('JDBC_HOSTNAME')
JDBC_DATABASE = os.getenv('jdbc_database') or os.getenv('JDBC_DATABASE')
JDBC_USERNAME = os.getenv('jdbc_username') or os.getenv('JDBC_USERNAME')
JDBC_PASSWORD = os.getenv('jdbc_password') or os.getenv('JDBC_PASSWORD')

# Trava de segurança: Verifica se as senhas foram achadas ANTES de configurar o Spark
faltantes = [
    nome for nome, valor in {
        'CLIENT_ID': CLIENT_ID,
        'TENANT_ID': TENANT_ID,
        'CLIENT_SECRET': CLIENT_SECRET,
        'STORAGE_ACCOUNT_NAME': STORAGE_ACCOUNT_NAME,
    }.items()
    if not valor
]

if faltantes:
    raise Exception(f'Variáveis obrigatórias ausentes no .env: {faltantes}')

# ==============================================================================
# INJEÇÃO DAS CREDENCIAIS NO SPARK (Usando as variáveis do .env)
# ==============================================================================

# 1. Avisa o Spark para usar o Service Principal (OAuth)
#spark.conf.set(f"fs.azure.account.auth.type.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", "OAuth")
#spark.conf.set(f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")

# 2. Passa as credenciais corretas lidas do .env para o Spark

#spark.conf.set(f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", CLIENT_ID)
#spark.conf.set(f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", CLIENT_SECRET)
#spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net", f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token")

print("Credenciais OAuth2 injetadas globalmente na sessão Spark do Databricks.")

print("Credenciais do Data Lake injetadas com sucesso no Spark!")
print('Configuração carregada com sucesso.')
print(f'Storage Account: {STORAGE_ACCOUNT_NAME}')
print(f'Container RAW: {CONTAINER_RAW}')
print(f'Container Squad1: {CONTAINER_SQUAD1}')
print(f'Pasta RAW: {RAW_FOLDER}')

## Clientes ADLS e opções para `deltalake`

In [0]:
credential = ClientSecretCredential(
    tenant_id=TENANT_ID,
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET
)

service_client = DataLakeServiceClient(
    account_url=f'https://{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net',
    credential=credential
)

file_system_client = service_client.get_file_system_client(file_system=CONTAINER_RAW)
file_system_squad1 = service_client.get_file_system_client(file_system=CONTAINER_SQUAD1)

# Opções usadas pelo pacote deltalake / delta-rs.
STORAGE_OPTIONS = {
    "account_name": STORAGE_ACCOUNT_NAME,
    "client_id": CLIENT_ID,
    "client_secret": CLIENT_SECRET,
    "tenant_id": TENANT_ID
}

print("Dicionário STORAGE_OPTIONS criado com sucesso!")

print('Clientes ADLS criados com sucesso.')

##  JDBC SQL Server

In [0]:
if JDBC_HOSTNAME and JDBC_DATABASE:
    jdbc_url = (
        f'jdbc:sqlserver://{JDBC_HOSTNAME}:1433;'
        f'database={JDBC_DATABASE};'
        'encrypt=true;'
        'trustServerCertificate=false;'
        'hostNameInCertificate=*.database.windows.net;'
        'loginTimeout=30;'
    )

    connection_properties = {
        'user': JDBC_USERNAME,
        'password': JDBC_PASSWORD,
        'driver': 'com.microsoft.sqlserver.jdbc.SQLServerDriver'
    }

    print('Configuração JDBC carregada.')
else:
    jdbc_url = None
    connection_properties = None
    print('JDBC não configurado neste ambiente.')

##  Funções para pastas no ADLS

In [0]:
PASTAS_RAIZ_PERMITIDAS = {'bronze', 'silver', 'gold', 'controle', 'staging'}


def normalizar_caminho_adls(caminho: str) -> str:
    if caminho is None:
        raise Exception('Caminho ADLS não pode ser None.')

    caminho = str(caminho).strip().strip('/')

    if not caminho:
        raise Exception('Caminho ADLS não pode ser vazio.')

    # Segurança: evita criar acidentalmente uma entidade na raiz, como ecommerce_itens_pedido.
    if '/' not in caminho and caminho not in PASTAS_RAIZ_PERMITIDAS:
        raise Exception(
            f"Caminho inseguro na raiz do container: '{caminho}'. "
            "Use camada/entidade, exemplo: bronze/ecommerce_rastreamento."
        )

    return caminho


def criar_pasta_adls(caminho: str, container: str = CONTAINER_SQUAD1):
    """
    Cria uma pasta/diretório no ADLS Gen2 caso ainda não exista.

    Exemplos válidos:
    - criar_pasta_adls('bronze')
    - criar_pasta_adls('bronze/ecommerce_endereco')

    Exemplo bloqueado:
    - criar_pasta_adls('ecommerce_endereco')
    """
    caminho = normalizar_caminho_adls(caminho)
    fs = service_client.get_file_system_client(file_system=container)

    try:
        fs.create_directory(caminho)
        print(f'Pasta criada: {container}/{caminho}')
    except Exception as e:
        msg = str(e).lower()
        if 'pathalreadyexists' in msg or 'already exists' in msg or 'resource already exists' in msg:
            print(f'Pasta já existe: {container}/{caminho}')
        else:
            raise


def criar_estrutura_medallion(entidades=None):
    if entidades is None:
        entidades = [
            'ecommerce_clientes',
            'ecommerce_itens_pedido',
            'ecommerce_pedidos',
            'ecommerce_itens_pedido',
            'ecommerce_rastreamento',
        ]

    for camada in ['bronze', 'silver', 'gold', 'controle', 'staging']:
        criar_pasta_adls(camada)

    for entidade in entidades:
        criar_pasta_adls(f'bronze/{entidade}')
        criar_pasta_adls(f'silver/{entidade}')

    criar_pasta_adls('gold/dq_resumo_por_regra')
    criar_pasta_adls('gold/dq_resumo_por_tabela')
    criar_pasta_adls('controle/dq_monitoring_logs')


def listar_adls(path: str = '', container: str = CONTAINER_SQUAD1, recursive: bool = True):
    fs = service_client.get_file_system_client(file_system=container)
    paths = fs.get_paths(path=path if path else None, recursive=recursive)

    for p in paths:
        tipo = 'DIR ' if p.is_directory else 'FILE'
        print(f'{tipo} | {p.name}')

##  Funções SQL Server opcionais

In [0]:
def escrever_sqlserver_gold(df_spark, schema: str, tabela: str, modo: str = "append"):
    """
    Replica DataFrame Spark para SQL Server.
    Use principalmente na camada Gold para consumo via Looker.
    """
    if not JDBC_HOSTNAME or not JDBC_DATABASE:
        print("SQL Server não configurado. Escrita ignorada.")
        return

    tabela_destino = f"{schema}.{tabela}"

    (
        df_spark.write
        .format("sqlserver")
        .mode(modo)
        .option("host", JDBC_HOSTNAME)
        .option("port", "1433")
        .option("database", JDBC_DATABASE)
        .option("user", JDBC_USERNAME)
        .option("password", JDBC_PASSWORD)
        .option("dbtable", tabela_destino)
        .option("encrypt", "true")
        .option("trustServerCertificate", "false")
        .save()
    )

    print(f"Tabela enviada para SQL Server: {tabela_destino}")

##  Testes rápidos

In [0]:
# pastas no diretorio 
print("Containers disponíveis:")
for fs in service_client.list_file_systems():
    print("-", fs.name)

print("\nPastas atuais em squad1:")
listar_adls(container=CONTAINER_SQUAD1, recursive=False)

In [0]:
# arquivos gerados diretamente em squad1

display(
    spark.sql("SHOW TABLES IN squad1")
)

In [0]:
print("===== TABELAS DO CATALOGO =====")
spark.sql("SHOW TABLES IN squad1").show(truncate=False)

print("\n===== ARQUIVOS NO CONTAINER SQUAD1 =====")

file_system = service_client.get_file_system_client("squad1")

for item in file_system.get_paths(recursive=True):
    print(item.name)